# 8장 - DuckDuckGo 검색

원본 파일: `chap08/duckduckgo_search.py`

In [2]:
import requests
from bs4 import BeautifulSoup
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from dotenv import load_dotenv
import os

load_dotenv()

model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv('OPENAI_API_KEY'),
)

QUESTION = "2025년 현대자동차 미국 시장 전망은 어떻게 되나요?"

In [3]:
def search_with_retry(search_tool, query, retries=3):
    """ddgs 라이브러리가 LibreSSL 환경에서 간헐적으로 TLS 버전 오류를 내므로 재시도"""
    last_error = None
    for _ in range(retries):
        try:
            return search_tool.invoke(query)
        except ValueError as e:
            last_error = e
    raise last_error

In [4]:
search = DuckDuckGoSearchResults(results_separator=';\n')
docs = search_with_retry(search, QUESTION)
print("===== 1. 기본 검색 결과 =====")
print(docs)

# 2) API wrapper로 검색 옵션(지역, 기간, 뉴스 소스 등) 지정
wrapper = DuckDuckGoSearchAPIWrapper(region="kr-kr", time="w")
news_search = DuckDuckGoSearchResults(
    api_wrapper=wrapper,
    source="news",  # 뉴스 소스로만 한정
    results_separator=';\n',
)
news_docs = search_with_retry(news_search, QUESTION)
print("\n===== 2. 뉴스 소스 한정 검색 =====")
print(news_docs)

===== 1. 기본 검색 결과 =====
snippet: Apr 1, 2025 · 현대차·기아는 2025년 미국 시장에서 사상 최대 실적을 목표로 설정하고, 전기차와 하이브리드 시장에서 강력한 성장세를 이어가고 있습니다 . 이들은 특히 차량 모델 경쟁력을 높이기 위해 공격적인 신차 출시와 마케팅 전략을 통해 판매 목표를 달성하고자 합니다., title: 2025년 현대차·기아의 미국 시장 정복 전략: 전기차와 하이브리드가 ..., link: https://seo.goover.ai/report/202504/go-public-report-ko-deb4e471-2938-494b-b19c-6eb5e10b1286-0-0.html;
snippet: Aug 5, 2025 · 4월부터 부과된 주요국에 대한 미국의 수입차 관세율이 기존 25%에서 평균 15%로 인하되었지만, 미국 수입업체들 입장에서는 신규 부담 업체별로 판가 인상을 통해 관세 비용을 전가하려는 시도. 전기차 보조금 폐지와 함께 소비자 부담 증가로 일시적 수요 위축 요인, title: 미국 자동차 판매 동향(2025년 7월), link: https://file.alphasquare.co.kr/media/pdfs/market-report/자동차현대차/기아의20250805하나증권.;
snippet: Mar 26, 2025 · 트럼프 행정부가 2024년 11월 재집권 이후 자동차 수입품에 25% 관세 부과를 예고함에 따라, 현대차 는 현지 생산 비중을 45%에서 75%로 끌어올려 관세 부담을 회피하는 전략을 채택했다. 이는 2023년 미국 내 판매 차량 중 58%가 수입차였던 점을 고려한 결정이다., title: 현대자동차그룹의 미국 투자 확대 전략 분석, link: https://rainbowwave.tistory.com/entry/현대자동차그룹의-미국-투자-확대-전략-분석;
snippet: Jun 23, 2025 · 무조건적인 비관도, 섣부른 낙관도 아닌 냉철한 분석을 통해 시장 상황에 맞는 현명한 투자 전략을

In [5]:
def get_article_text(url: str) -> str:
    """주어진 URL에서 기사 본문 텍스트만 추출"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        article = soup.find('article')
        if article:
            return article.get_text(strip=True)

        div = soup.find('div', id="CmAdContent")
        if div:
            return div.get_text(strip=True)

        return "기사 내용을 찾을 수 없습니다."
    except requests.exceptions.RequestException as e:
        return f"URL을 가져오는 중 오류 발생: {e}"

In [6]:
links = []
for doc in news_docs.split(';\n'):
    link = doc.split('link:')[-1].strip()
    links.append(link)

print("\n===== 3. 추출된 링크 =====")
print(links)

# 4) 각 링크에서 기사 본문 가져오기
articles = []
for link in links:
    article_text = get_article_text(link)
    articles.append(article_text)

print("\n===== 4. 기사 본문 =====")
for link, text in zip(links, articles):
    print(f"URL: {link}")
    print(f"내용: {text[:200]}...")
    print('-' * 50)

# 5) 검색 결과(context)를 바탕으로 GPT가 답변 생성 (RAG 패턴)
question_answering_prompt = ChatPromptTemplate([
    ("system", "사용자의 질문에 대해 아래 context에 기반하여 답변하라.:\n\n{context}"),
    MessagesPlaceholder(variable_name="messages"),
])
document_chain = question_answering_prompt | model

chat_history = InMemoryChatMessageHistory()
chat_history.add_user_message(QUESTION)

answer = document_chain.invoke({
    "messages": chat_history.messages,
    "context": docs,
})
chat_history.add_ai_message(answer)

print("\n===== 5. 검색 기반 답변 =====")
print(answer.content)


===== 3. 추출된 링크 =====
['https://www.hyundaimotorgroup.com/ko/news/byline-45', 'https://car.withnews.kr/economy/us-car-demand-hyundai', 'https://cartech.nate.com/content/2288838', 'https://www.g-enews.com/article/Global-Biz/2026/06/2026062918545935009a1f309431_1']

===== 4. 기사 본문 =====
URL: https://www.hyundaimotorgroup.com/ko/news/byline-45
내용: 기사 내용을 찾을 수 없습니다....
--------------------------------------------------
URL: https://car.withnews.kr/economy/us-car-demand-hyundai
내용: Home»경제“현대차 내연차까지 비상 걸렸다”…미국 신차 5대 중 1대 ‘이 지경’에 ‘대반전’박수진 기자댓글 1입력2026.06.30 08:00현대차 / 출처 : 연합뉴스미국 신차 시장의 연간 판매량이 오는 2040년까지 200만 대 이상 급감할 수 있다는 경고등이 켜지면서 글로벌 완성차 업계에 긴장감이 돌고 있다.글로벌 컨설팅업체 베인앤드컴퍼니는 최근 ...
--------------------------------------------------
URL: https://cartech.nate.com/content/2288838
내용: 현대차그룹, 美서 포드 턱밑까지 추격…하이브리드로 ‘3위 고지’ 눈앞더구루|나신혜 기자|2026.06.30...
--------------------------------------------------
URL: https://www.g-enews.com/article/Global-Biz/2026/06/2026062918545935009a1f309431_1
내용: 기사 내용을 